# Inspect e2e training batch (`.npz`)

Written by `lingua_modified/tests/apps/main/test_batch_generation_end2end.py` when tests run (e.g. `scripts/run_tests.sh`). The archive contains `batch` and `sampled_batches`; a `.meta.json` may sit next to the file.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np

# If discovery fails, set a path and re-run this cell.
# MANUAL_NPZ: Path | None = Path("…/e2e_training_batch.npz")
MANUAL_NPZ: Path | None = None

# Resolve location: notebook may be run with cwd = repo root or .test-artifacts/
def find_e2e_npz() -> Path:
    cwd = Path.cwd()
    direct = [
        cwd / "basetemp/test_batch_generation_end2end0/e2e_training_batch.npz",
        cwd / ".test-artifacts/basetemp/test_batch_generation_end2end0/e2e_training_batch.npz",
    ]
    for p in direct:
        if p.is_file():
            return p.resolve()
    roots = [cwd, cwd / ".test-artifacts", cwd.parent]
    for root in roots:
        if root.is_dir():
            for p in root.rglob("e2e_training_batch.npz"):
                return p.resolve()
    raise FileNotFoundError(
        "e2e_training_batch.npz not found. Run tests (e.g. ./scripts/run_tests.sh) or set MANUAL_NPZ in the cell above."
    )


NPZ_PATH = MANUAL_NPZ if MANUAL_NPZ is not None else find_e2e_npz()
META_PATH = NPZ_PATH.with_suffix(".meta.json")
print("npz:", NPZ_PATH)
print("meta:", META_PATH if META_PATH.is_file() else "(none)")

npz: /Users/kirillzemlanskij/Workspace/PhysicsLM4/.test-artifacts/basetemp/test_batch_generation_end2end0/e2e_training_batch.npz
meta: /Users/kirillzemlanskij/Workspace/PhysicsLM4/.test-artifacts/basetemp/test_batch_generation_end2end0/e2e_training_batch.meta.json


In [3]:
with np.load(NPZ_PATH) as z:
    print("Keys:", list(z.keys()))
    batch = np.ascontiguousarray(z["batch"])
    sampled = int(np.asarray(z["sampled_batches"]).ravel()[0])
print("sampled_batches:", sampled)
print("batch array:", batch.shape, batch.dtype, batch.nbytes, "bytes")

# Same layout as train: batch[:, :, 0] = input_ids, batch[:, :, 1] = labels
D_INPUT = 0
D_LABELS = 1
if META_PATH.is_file():
    meta = json.loads(META_PATH.read_text(encoding="utf-8"))
    D_INPUT = int(meta.get("input_ids_slice_dim", 0))
    D_LABELS = int(meta.get("labels_slice_dim", 1))
    print("\nMeta:")
    for k, v in meta.items():
        print(f"  {k}: {v}")
else:
    meta = {}

PAD_TOKEN = int(meta.get("pad_token", 0))
IGNORE_LABEL = int(meta.get("no_train_label_token", -100))

input_ids = batch[:, :, D_INPUT]
labels = batch[:, :, D_LABELS]
bsz, seqlen = input_ids.shape
print(f"\ninput_ids: {input_ids.shape}  |  labels: {labels.shape}")
print(f"non-padding tokens (row 0, pad={PAD_TOKEN}): {(input_ids[0] != PAD_TOKEN).sum()} / {seqlen}")
print(f"trainable label positions in row 0 (ignore={IGNORE_LABEL}): {(labels[0] != IGNORE_LABEL).sum()} / {seqlen}")
np.set_printoptions(threshold=120, edgeitems=6, linewidth=200)
print("\ninput_ids[0, :64]:\n", input_ids[0, :64])
print("\nlabels[0, :64]:\n", labels[0, :64])

Keys: ['batch', 'sampled_batches']
sampled_batches: 1
batch array: (2, 1024, 2) int64 32768 bytes

Meta:
  batch_shape: [2, 1024, 2]
  batch_dtype: int64
  seq_len: 1024
  batch_size: 2
  pad_token: 0
  no_train_label_token: -100
  input_ids_slice_dim: 0
  labels_slice_dim: 1

input_ids: (2, 1024)  |  labels: (2, 1024)
non-padding tokens (row 0, pad=0): 101 / 1024
trainable label positions in row 0 (ignore=-100): 14 / 1024

input_ids[0, :64]:
 [100  19   6   2  19   2  20  10  12  20   8  12   3   3   3  18   3  12  14   2  20  10  12   2  14   4   4  19   4   9   4  12   6   2  19  13   4   9   4  12   3   3   3  18   8  12  14   2  14
   4   4  19  11  20  13  11   3  12  19 208   6   2  19   2]

labels[0, :64]:
 [-100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100
 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -100 -10

In [11]:
input_ids[0,:]

array([100,  19,   6,   2,  19,   2, ...,   0,   0,   0,   0,   0,   0], shape=(1024,))

In [14]:
labels[0,:]

array([-100, -100, -100, -100, -100, -100, ..., -100, -100, -100, -100, -100, -100], shape=(1024,))